# tiny-yolov3-visdrone sur visdrone — entraînement

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lrivals/EmbeddedComputervision/blob/main/notebooks/visdrone/tiny-yolov3-visdrone_train.ipynb)

Rôle : entraînement ; palier **N** (4000 itérations par défaut, plusieurs heures sur CPU) ; évaluation finale sur `SUBSET` images. Tâches : [T11.5](../../docs/tasks/M11-jeux-de-donnees.md) ;
notebooks : [M14](../../docs/tasks/M14-notebooks.md).

Prérequis :
- données : `data/visdrone/VisDrone2019-DET-val/annotations` — tools/get_datasets.sh visdrone (inscription : source affichée)
- poids : `weights/yolov3-tiny.weights` — tools/get_weights.sh

Chaque étape affiche la commande `python tools/…` qu'elle lance : un résultat se
reproduit en ligne de commande. Sorties dans `build/notebooks/visdrone/tiny-yolov3-visdrone/`, jamais dans `results/`
(une mAP sur `SUBSET` images n'est pas publiable, règles de M12). Notebook généré par
`python -m tools.notebooks` (`make notebooks`) : modifier le gabarit
`tools/notebooks/gabarits.py`, pas ce fichier.

## Paramètres

In [ ]:
NET          = 'build/m11/cfg/tiny-yolov3-visdrone.cfg'  # nom de CFG_FILES ou cfg (relative à la racine)
DATASET      = 'visdrone'  # clé de DATASETS
SPLIT        = None  # None : split d'évaluation du jeu
WEIGHTS      = 'build/notebooks/visdrone/tiny-yolov3-visdrone/final.weights'  # poids produits, évalués à la fin
DEVICE       = 'gpu'  # cpu | gpu (entraînement seulement ; le reste sur CPU)
SUBSET       = 50  # images évaluées ; 0 : split complet (palier N)
SIZE         = None  # entrée LxH (ex. '640x192') ; None : celle de la cfg
RESIZE       = 'stretch'  # stretch (référence M12) | letterbox
METRIC       = None  # None : celle du jeu (coco pour coco et flir)
DATA_ROOT    = None  # racine du jeu ; None : data/<jeu> (Colab : dossier Drive)
BASE         = 'tiny-yolov3-voc'  # base de make_cfg.py
CHANNELS     = None  # canaux de la cfg (1 : thermique)
ANCHORS_K    = 6  # ancres de la cfg (6 : Tiny-YOLOv3)
INIT         = 'coco'  # coco | he | chemin d'un .weights
INIT_NET     = None  # réseau des poids de INIT s'il diffère de NET
ITERS        = 6000  # itérations (palier N au-delà de 600)
BATCH        = 16
RESIZE_TRAIN = 'stretch'  # letterbox | stretch : géométrie à l'entraînement (stretch : celle de l'éval, RESIZE)
CROP         = 0  # découpes N×N px de l'image d'origine (tuiles) ; 0 : image entière
STEPS        = '4800,5400'  # itérations où le LR baisse (ex. '4800,5400')
SCALES       = '0.1,0.1'  # facteurs du LR à STEPS (ex. '0.1,0.1')
LR           = 0.001
BURN_IN      = 500
MULTISCALE   = True  # 320-608 tous les 10 lots
SAVE_EVERY   = None  # None : défaut de train.py (200)
WORKERS      = 4
QAT          = ''  # schéma wXaY (ex. 'w4a4') : QAT (T9.3, T12.5)
QAT_STEPS    = None  # steps.json de quant_lowbit.py (PTQ)
ADMM         = None  # plan JSON de poids ({} : mixed6 partout, T9.2, T12.6)
ADMM_RHO     = 0.001
ADMM_GROWTH  = 1.3
ADMM_EVERY   = 100
DRIVE_DIR    = '/content/drive/MyDrive/EmbeddedCV'  # Colab : archives data/<jeu>.tar, sorties et reprise dans runs/ ; None : pas de Drive
JOBS         = None  # processus des évaluations ; None : un par cœur CPU, plafonné par la mémoire (runtime TPU de Colab : des dizaines)
SHOW         = 6  # images affichées
OUT          = 'build/notebooks/visdrone/tiny-yolov3-visdrone'  # sorties du notebook
REPO_URL     = 'https://github.com/lrivals/EmbeddedComputervision.git'  # Colab : dépôt cloné
REV          = 'main'  # Colab : révision

## Environnement

Local : rien n'est installé ni téléchargé, la cellule vérifie les prérequis et
s'arrête avec la commande à lancer. Colab : clone du dépôt, installation, CuPy si
`DEVICE = "gpu"`, poids, puis données par `colab.prepare_data` (archive
`<DRIVE_DIR>/data/<jeu>.tar` ou téléchargement direct ; ExDark et FLIR : archive
seulement, faite sur le PC par `tools/get_datasets.sh pack <jeu>`).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

COLAB = "google.colab" in sys.modules


def repo_root():
    for d in (Path.cwd(), *Path.cwd().parents):
        if (d / "tools" / "notebooks" / "commandes.py").exists():
            return d
    return None


ROOT = repo_root()
if COLAB:
    if ROOT is None:
        ROOT = Path("/content/EmbeddedComputervision")
        if not ROOT.exists():
            subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    # Clone d'une session précédente : remis à REV (notebook et code du même commit).
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin", REV], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "FETCH_HEAD"], check=True)
    for m in [m for m in sys.modules if m.split(".")[0] in ("tools", "yolo")]:
        del sys.modules[m]  # modules importés avant la mise à jour
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                    f"{ROOT}/python[data,plots]"], check=True)
    if DEVICE == "gpu":
        # Runtime CPU : pas de pilote, CuPy dirait « cudaErrorInsufficientDriver ».
        try:
            smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        except FileNotFoundError:
            smi = None
        if smi is None or smi.returncode != 0:
            raise RuntimeError("runtime Colab sans GPU : créer un nouveau serveur Colab "
                               "de type GPU (T4, L4 ou A100), ou DEVICE = 'cpu'")
        cuda = smi.stdout.split("CUDA Version:")[-1].split()[0]  # version du pilote
        cupy_pkg = "cupy-cuda13x" if int(cuda.split(".")[0]) >= 13 else "cupy-cuda12x"
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", cupy_pkg],
                       check=True)
elif ROOT is None:
    raise RuntimeError("racine du dépôt introuvable : ouvrir le notebook depuis notebooks/")
os.chdir(ROOT)
for p in (ROOT, ROOT / "python"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from tools.notebooks import commandes as C
from tools.notebooks import env

TRACE = env.trace(DEVICE)
env.check_device(DEVICE)
env.prepare(DATASET, INIT if str(INIT).endswith('.weights') else 'weights/yolov3-tiny.weights' if INIT == 'coco' else (), None, DATA_ROOT, COLAB, hint='',
            drive_dir=DRIVE_DIR)
Path(OUT).mkdir(parents=True, exist_ok=True)

## Préparation (jeux hors VOC)

Ancres k-means sur le split d'entraînement (`tools/kmeans_anchors.py`, table dans
`OUT`), puis cfg à N classes (`tools/make_cfg.py`), comme `tools/m11.sh <jeu>-prep`.
Sautée si la cfg existe déjà.

In [ ]:
import csv
import json

import numpy as np
from IPython.display import Markdown, display
from PIL import Image

from tools.detect import draw
from tools.notebooks.matrice import estimate
from yolo.data import datasets
from yolo.data.letterbox import input_size, parse_size, size_label
from yolo.models.tiny_yolo import load_cfg

if not str(NET).endswith(".cfg"):
    print(f"{NET} : cfg du dépôt, pas de préparation")
elif Path(NET).exists():
    print(f"{NET} existe : préparation sautée")
else:
    anchors = Path(OUT) / f"anchors_{DATASET}{'_' + str(SIZE) if SIZE else ''}.md"
    C.run(C.cmd_anchors(DATASET, SIZE, anchors, DATA_ROOT))
    C.run(C.cmd_make_cfg(BASE, DATASET, C.anchors_k(anchors, ANCHORS_K), NET, SIZE,
                         CHANNELS))

## Aperçu des cibles

Quelques images augmentées (`yolo.data.loader.VOCDataset`, donc `yolo.data.augment`)
avec leurs boîtes : contrôle des annotations avant des heures de calcul.

In [ ]:
cfg = load_cfg(NET)
size = input_size(cfg, parse_size(SIZE) if SIZE else None)
channels = cfg["input"][0]
names = datasets.DATASETS[DATASET].classes
samples = datasets.load(DATASET, DATA_ROOT, role="train")
print(f"{len(samples)} images d'entraînement, entrée {size_label(size)}, {channels} canal(aux)")
from yolo.data.loader import VOCDataset

preview = VOCDataset(samples[:SHOW], train=True, channels=channels,
                     resize=RESIZE_TRAIN, crop=CROP)
for k in range(len(preview)):
    chw, boxes, labels = preview.load(k, size, np.random.default_rng(k))
    hwc = (np.clip(chw.transpose(1, 2, 0), 0, 1) * 255).astype(np.uint8)
    img = Image.fromarray(hwc[..., 0] if channels == 1 else hwc)
    display(draw(img, boxes, np.ones(len(labels)), labels, names))

## Affinage

`tools/train.py` dans un sous-processus dont la sortie est relayée ligne à ligne ; une
interruption du noyau laisse le dernier `checkpoint.npz` (tous les `SAVE_EVERY`), et
relancer la cellule reprend (`--resume`). Sur Colab avec `DRIVE_DIR`, `OUT` est copié
dans `<DRIVE_DIR>/runs/` toutes les 10 min et en fin de cellule
(`colab.sync_outputs`), puis restauré en début de session (`colab.restore_outputs`) :
une session coupée reprend au dernier checkpoint copié, sur le même backend
(T12.11, T14.8).


### Runs longs VisDrone (M15)

Un run par dossier : régler `OUT` sur `RUNS_DIR/runs/<nom>` pour que le notebook
`_infer` le trouve (`RUN = '<nom>'`, `COMPARE`), par exemple :

| run | réglages |
|---|---|
| `long-416` | défauts : `RESIZE_TRAIN = 'stretch'`, `CROP = 0` |
| `long-416-crop640` | `CROP = 640` ; à évaluer avec `TILES = 640` dans `_infer` |
| `long-608` | `SIZE = '608'` et `NET` = cfg à 608 (`tools/m11.sh visdrone-prep` avec `SIZE=608`) |

`RESIZE_TRAIN = 'stretch'` aligne l'entraînement sur l'éval (`RESIZE`) : en letterbox,
les objets sont 1,3 à 1,8 fois moins hauts qu'à l'éval. Les ancres de la cfg restent
celles du letterbox : tous les runs s'évaluent avec la même cfg.

In [ ]:
import threading

from tools.notebooks import colab, runs

RUN = Path(OUT)
SYNC = COLAB and bool(DRIVE_DIR)
if SYNC:  # reprise après coupure de session : sorties de la session précédente
    env.mount_drive(DRIVE_DIR)
    colab.restore_outputs(OUT, DRIVE_DIR)
RUN.mkdir(parents=True, exist_ok=True)
if ADMM and not Path(ADMM).exists():
    Path(ADMM).parent.mkdir(parents=True, exist_ok=True)
    Path(ADMM).write_text("{}\n")  # mixed6 partout, comme tools/m12.sh admm
seconds = estimate(ITERS, BATCH, DEVICE, RUN / "loss.csv")
print(f"{ITERS} itérations × {BATCH} images : "
      + (f"≈ {seconds / 3600:.1f} h sur {DEVICE}" if seconds else
         f"durée non mesurée sur {DEVICE} (T12.11-e)"))
resume = (RUN / "checkpoint.npz").exists()
runs.write_meta(RUN, batch=BATCH, subset=0, iters=ITERS, lr=LR, device=DEVICE,
                net=NET, dataset=DATASET, qat=QAT, admm=bool(ADMM), resize=RESIZE_TRAIN,
                crop=CROP, rev=TRACE["rev"])
stop = threading.Event()


def sync_loop(every=600):  # copie sur Drive toutes les 10 min, checkpoint compris
    while not stop.wait(every):
        colab.sync_outputs(OUT, DRIVE_DIR)


if SYNC:
    threading.Thread(target=sync_loop, daemon=True).start()
try:
    C.run(C.cmd_train(NET, RUN, DATASET, INIT, INIT_NET, resume, QAT, QAT_STEPS, ADMM,
                      ADMM_RHO, ADMM_GROWTH, ADMM_EVERY, ITERS, BATCH, LR, BURN_IN,
                      MULTISCALE, None, 0, SAVE_EVERY, WORKERS, DEVICE, DATA_ROOT,
                      resize=RESIZE_TRAIN, crop=CROP, steps=STEPS, scales=SCALES),
          log=RUN / "log.txt")
finally:
    stop.set()
    if SYNC:
        colab.sync_outputs(OUT, DRIVE_DIR)

## Courbes

Perte et taux d'apprentissage de `loss.csv` (mode automatique de M13) ; relancer la
cellule pendant un entraînement long redessine jusqu'au dernier point écrit.

In [ ]:
C.run(C.cmd_figures("train", RUN))
for png in sorted((RUN / "figures").glob("*.png")):
    display(Image.open(png))

## Évaluation

`final.weights` par `tools/eval_voc.py` (gabarit `infer`, T14.3) ; en QAT ou ADMM,
modèle basse précision par `tools/quant_lowbit.py` puis `tools/eval_quant.py`.

In [ ]:
if QAT or ADMM:
    C.run(C.cmd_quant_lowbit(NET, RUN / "checkpoint.npz", RUN / "model", QAT, bool(ADMM)))
    result = RUN / "map.json"
    C.run(C.cmd_eval_quant(NET, result, "int", SUBSET, JOBS, DATASET,
                           model_dir=RUN / "model", metric=METRIC, split=SPLIT,
                           size=SIZE, data_root=DATA_ROOT))
    d = json.loads(result.read_text())
    print(f"mAP entière {100 * d['results']['int']['map']:.2f} ({d['images']} images)")
else:
    result = RUN / "map_float.md"
    result.unlink(missing_ok=True)
    C.run(C.cmd_eval_voc(NET, RUN / "final.weights", DATASET, RESIZE, SUBSET, METRIC,
                         SPLIT, SIZE, DATA_ROOT, out=RUN / "dets", markdown=result,
                         jobs=JOBS, tiles=CROP))
    display(Markdown(result.read_text()))

## Vitesse du backend (T12.11-e)

Secondes par image des dernières itérations (`loss.csv`) ; le lot maximal qui tient
en mémoire GPU se trouve en relançant avec `BATCH` croissant. Un chiffre issu de
poids entraînés sur GPU indique le backend.

In [ ]:
rows = list(csv.DictReader((RUN / "loss.csv").open())) if (RUN / "loss.csv").exists() else []
if rows:
    last = rows[-50:]
    s = sum(float(r["seconds"]) for r in last) / len(last) / BATCH
    print(f"{s:.3f} s/image sur {DEVICE}, lot {BATCH}, {len(rows)} itérations journalisées")

## Résumé

In [ ]:
print(f"sorties : {RUN}/ (checkpoint.npz, loss.csv, final.weights, figures/) ; "
      f"backend {DEVICE} ; révision {TRACE['rev']}")
print(f"évaluation : {result}")
print("commandes équivalentes :")
for c in C.HISTORY:
    print("  " + c)